In [1]:
#importing libraries
import tensorflow as tf
from tensorflow.keras import layers, models
import numpy as np
import matplotlib.pyplot as plt

In [4]:
import pandas as pd
# Step 1: Load the MNIST dataset
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()

# Step 2: Print data size and shape
print("Training data shape:", x_train.shape)
print("Test data shape:", x_test.shape)
print("Number of training samples:", len(x_train))
print("Number of test samples:", len(x_test))

# Step 3: Save data to CSV
x_train_flat = x_train.reshape(x_train.shape[0], -1)
x_test_flat = x_test.reshape(x_test.shape[0], -1)

train_data = np.column_stack((y_train, x_train_flat))
test_data = np.column_stack((y_test, x_test_flat))

columns = ['label'] + [f'pixel{i}' for i in range(x_train_flat.shape[1])]

train_df = pd.DataFrame(train_data, columns=columns)
test_df = pd.DataFrame(test_data, columns=columns)

train_df.to_csv('mnist_train.csv', index=False)
test_df.to_csv('mnist_test.csv', index=False)

print("CSV files saved: mnist_train.csv and mnist_test.csv")

Training data shape: (60000, 28, 28)
Test data shape: (10000, 28, 28)
Number of training samples: 60000
Number of test samples: 10000
CSV files saved: mnist_train.csv and mnist_test.csv


In [11]:
# Step 4: Preprocessing - Resize to 32x32, normalize, expand dims for channels
IMG_SIZE = 32
x_train_resized = tf.image.resize_with_pad(tf.expand_dims(x_train, -1), IMG_SIZE, IMG_SIZE)
x_test_resized = tf.image.resize_with_pad(tf.expand_dims(x_test, -1), IMG_SIZE, IMG_SIZE)

# Normalize to [0, 1]
x_train_norm = x_train_resized / 255.0
x_test_norm = x_test_resized / 255.0

In [12]:
# Step 5: One-hot encode labels
num_classes = 10
y_train_ohe = tf.keras.utils.to_categorical(y_train, num_classes)
y_test_ohe = tf.keras.utils.to_categorical(y_test, num_classes)

In [21]:
# Step 6: Data Augmentation (for training data)
from tensorflow.keras.preprocessing.image import ImageDataGenerator
datagen = ImageDataGenerator(
    rotation_range=10,
    width_shift_range=0.1,
    height_shift_range=0.1,
    zoom_range=0.1
)

# Fit the data generator on the training set
datagen.fit(x_train_norm)

In [22]:
# Step 6: Build CNN model with Dropout
model = models.Sequential([
    layers.Conv2D(32, (3, 3), activation='relu', input_shape=(32,32, 1)),
    layers.MaxPooling2D((2, 2)),
    layers.Dropout(0.25),

    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),
    layers.Dropout(0.25),

    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(num_classes, activation='softmax')
])


In [23]:
# Step 7: Compile the model
model.compile(optimizer='adam',
              loss='categorical_crossentropy',
              metrics=['accuracy'])


In [24]:
# Step 8: Callback for early stopping only 
callbacks = [
    tf.keras.callbacks.EarlyStopping(patience=3, restore_best_weights=True)
]


In [26]:
# Step 9: Train the model
history = model.fit(
    x_train_norm, y_train_ohe,
    epochs=15,
    batch_size=64,
    validation_split=0.2,
    callbacks=callbacks
)

Epoch 1/15
750/750 ━━━━━━━━━━━━━━━━━━━━ 62s 82ms/step - accuracy: 0.9011 - loss: 0.3198 - val_accuracy: 0.9768 - val_loss: 0.0778
Epoch 2/15
750/750 ━━━━━━━━━━━━━━━━━━━━ 45s 59ms/step - accuracy: 0.9624 - loss: 0.1232 - val_accuracy: 0.9857 - val_loss: 0.0522
Epoch 3/15
750/750 ━━━━━━━━━━━━━━━━━━━━ 59s 78ms/step - accuracy: 0.9718 - loss: 0.0923 - val_accuracy: 0.9870 - val_loss: 0.0447
Epoch 4/15
750/750 ━━━━━━━━━━━━━━━━━━━━ 45s 59ms/step - accuracy: 0.9770 - loss: 0.0759 - val_accuracy: 0.9896 - val_loss: 0.0378
Epoch 5/15
750/750 ━━━━━━━━━━━━━━━━━━━━ 61s 82ms/step - accuracy: 0.9805 - loss: 0.0638 - val_accuracy: 0.9877 - val_loss: 0.0408
Epoch 6/15
750/750 ━━━━━━━━━━━━━━━━━━━━ 76s 74ms/step - accuracy: 0.9818 - loss: 0.0581 - val_accuracy: 0.9898 - val_loss: 0.0350
Epoch 7/15
750/750 ━━━━━━━━━━━━━━━━━━━━ 52s 69ms/step - accuracy: 0.9841 - loss: 0.0501 - val_accuracy: 0.9902 - val_loss: 0.0343
Epoch 8/15
750/750 ━━━━━━━━━━━━━━━━━━━━ 57s 76ms/step - accuracy: 0.9854 - loss: 0.0470 - 

In [27]:
# Step 11: Evaluate on test set
test_loss, test_accuracy = model.evaluate(x_test_norm, y_test_ohe)
print(f"\nTest Accuracy: {test_accuracy:.4f}")

313/313 ━━━━━━━━━━━━━━━━━━━━ 14s 40ms/step - accuracy: 0.9905 - loss: 0.0277

Test Accuracy: 0.9921


In [ ]:
# Step 12: Plot accuracy
plt.plot(history.history['accuracy'], label='Train Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.title('Training and Validation Accuracy')
plt.show()

SyntaxError: incomplete input (1358004525.py, line 8)